In [1]:
import os
import gc
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import timm
from tqdm import tqdm
from pathlib import Path

In [2]:
# 1. CONFIGURATION
class CFG:
    SR = 32000
    DURATION = 5
    CHUNK_SIZE = SR * DURATION
    INPUT_DIR = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes/")
   
   
    MODEL_PATH_B3_RARE_AMPH = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/rare-amphibian-oversample/1/bird_model_best.pth"
    MODEL_PATH_B0 = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/syntheticdata/8/bird_model_best.pth"
   
    NUM_CLASSES = 234
    THRESHOLD = 0.00
    B3_COMP_RARE_AMPH = 0.85
    B0_WEIGHT = 0.15  

for dirname, _, filenames in os.walk('/kaggle/input/competitions/birdclef-2026/test_soundscapes/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/birdclef-2026/test_soundscapes/readme.txt


In [3]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/kaggle/working/birdclef2026_s22_pseudolabels"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Output directory: {OUTPUT_DIR}")

Output directory: /kaggle/working/birdclef2026_s22_pseudolabels


In [4]:
# REFINED MODEL CLASS
class BirdModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', num_classes=234):
        super().__init__()
        # 1. Create model with 1 input channel instead of 3
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=1)
        
        # 2. Based on your error message, your trained weights use "fc" 
        # instead of "backbone.classifier". Let's map it:
        self.fc = nn.Linear(self.backbone.classifier.in_features, num_classes)
        
        # 3. Disable the default classifier so it doesn't conflict
        self.backbone.classifier = nn.Identity()
        
    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x) # Use the "fc" layer as in your training
        return x

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Load B3 Model trained on composite primary labels and no soundscape mixing (234 classes)
model_b3_rare_amph = BirdModel(model_name='efficientnet_b3', num_classes=234)
model_b3_rare_amph.load_state_dict(torch.load(CFG.MODEL_PATH_B3_RARE_AMPH, map_location=device))
model_b3_rare_amph.to(device).eval()


# Load B0 Model (206 classes)
model_b0 = BirdModel(model_name='efficientnet_b0', num_classes=206)
model_b0.load_state_dict(torch.load(CFG.MODEL_PATH_B0, map_location=device))
model_b0.to(device).eval()



Using device: cuda


BirdModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Con

In [6]:
# 4. LOAD SUBMISSION FORMAT + CREATE SPECIES MAPPING

# Load the official sample submission to get the 234 required species columns
sample_sub_path = Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)
target_species = sample_sub.columns[1:].tolist()

In [7]:
# Based on your sidebar screenshot, the data is here:
data_root = Path("/kaggle/input/competitions/birdclef-2026")

In [8]:
# Load full 234 taxonomy
taxonomy_path = Path("/kaggle/input/competitions/birdclef-2026/taxonomy.csv")
taxonomy_df = pd.read_csv(taxonomy_path)
all_species_234 = sorted(taxonomy_df['primary_label'].unique())
mapping_234 = {s: i for i, s in enumerate(all_species_234)}

print(f"✅ Mapping created for {len(mapping_234)} species.")
# Verification check:
if len(mapping_234) != 234:
    print(f"⚠️ Warning: Found {len(mapping_234)} species, expected 234!")

# Load the 206 training labels used for B0
# Assuming you have the train.csv available
train_df = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train.csv")
species_206 = sorted(train_df['primary_label'].unique())

# Create an index map: which index in B3 (234) corresponds to the index in B0 (206)?
b0_to_b3_idx = [mapping_234[s] for s in species_206]

✅ Mapping created for 234 species.


In [9]:
def get_spectrogram(audio_chunk):
    # Matches the 1-channel logic your model expects
    S = librosa.feature.melspectrogram(y=audio_chunk, sr=CFG.SR, n_mels=128, fmax=16000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    return S_dB

In [10]:
def get_ensemble_probs(file_path):
    y, _ = librosa.load(file_path, sr=CFG.SR)

    ensemble_probs = []
    chunk_npy_files = []

    for chunk_idx, i in enumerate(
        range(0, len(y), CFG.CHUNK_SIZE)
    ):
        chunk = y[i:i + CFG.CHUNK_SIZE]

        if len(chunk) < CFG.CHUNK_SIZE:
            chunk = np.pad(
                chunk,
                (0, CFG.CHUNK_SIZE - len(chunk))
            )

        spec = get_spectrogram(chunk)

        # -----------------------------------
        # Save spectrogram for pseudo labels
        # -----------------------------------
        npy_name = (
            f"{file_path.stem}"
            f"__{chunk_idx:04d}.npy"
        )

        np.save(
            OUTPUT_DIR / npy_name,
            spec.astype(np.float16)
        )

        chunk_npy_files.append(npy_name)

        tensor_spec = (
            torch.from_numpy(spec)[None, None, ...]
            .float()
            .to(device)
        )

        with torch.no_grad():

            # B3
            out_b3_rare_amph = model_b3_rare_amph(
                tensor_spec
            )

            probs_b3_rare_amph = (
                torch.sigmoid(out_b3_rare_amph)
                .cpu()
                .numpy()[0]
            )

            # B0
            out_b0 = model_b0(tensor_spec)

            probs_b0_raw = (
                torch.sigmoid(out_b0)
                .cpu()
                .numpy()[0]
            )

            probs_b0_aligned = np.zeros(
                234,
                dtype=np.float32
            )

            for old_idx, new_idx in enumerate(
                b0_to_b3_idx
            ):
                probs_b0_aligned[new_idx] = (
                    probs_b0_raw[old_idx]
                )

            blended_probs = (
                probs_b3_rare_amph
                * CFG.B3_COMP_RARE_AMPH
                + probs_b0_aligned
                * CFG.B0_WEIGHT
            )

            ensemble_probs.append(blended_probs)

    del y

    return (
        np.array(ensemble_probs),
        chunk_npy_files
    )

In [11]:
def aggregate_and_format(
    raw_probs,
    file_id,
    target_species,
    species_mapping
):
    """
    Recall-oriented temporal smoothing + probability sharpening.

    Key ideas:
    --------------------------------
    1. Preserve weak detections
       (BirdCLEF rewards recall heavily)

    2. Use max-based smoothing
       (validated by leaderboard)

    3. Apply mild probability sharpening
       because model appears underconfident

    4. Keep submission-compatible formatting
    """

    raw_probs = raw_probs.astype(np.float32)

    num_chunks = len(raw_probs)

    # ------------------------------------------------------------
    # TEMPORAL SMOOTHING
    # ------------------------------------------------------------
    smoothed_probs = np.zeros_like(raw_probs)

    for i in range(num_chunks):

        # Slightly wider context window
        start = max(0, i - 2)
        end   = min(num_chunks, i + 3)

        window = raw_probs[start:end]

        # --------------------------------------------------------
        # Recall-oriented smoothing
        #
        # Max preserves faint / partial calls
        # Mean reduces extreme instability slightly
        # --------------------------------------------------------
        max_probs  = np.max(window, axis=0)
        mean_probs = np.mean(window, axis=0)

        smoothed_probs[i] = (
            0.85 * max_probs +
            0.15 * mean_probs
        )

    # ------------------------------------------------------------
    # PROBABILITY SHARPENING
    #
    # Your diagnostics suggest:
    # - AUC extremely high
    # - AP decent
    # - probabilities underconfident
    #
    # Power transform increases confidence separation
    # while mostly preserving ranking.
    # ------------------------------------------------------------
    smoothed_probs = np.power(smoothed_probs, 0.75)

    # ------------------------------------------------------------
    # FORMATTING
    # ------------------------------------------------------------
    file_results = []

    for i in range(num_chunks):

        row_id = f"{file_id}_{(i + 1) * 5}"

        pred_dict = {
            "row_id": row_id
        }

        chunk_probs = smoothed_probs[i]

        """
        # Optional geographic suppression
        chunk_probs = apply_geographic_suppression(
            chunk_probs,
            species_mapping
        )
        """

        # --------------------------------------------------------
        # Thresholding
        # --------------------------------------------------------
        chunk_probs = np.where(
            chunk_probs > CFG.THRESHOLD,
            chunk_probs,
            0.0
        )

        # --------------------------------------------------------
        # Populate submission row
        # --------------------------------------------------------
        for species in target_species:

            if species in species_mapping:

                pred_dict[species] = chunk_probs[
                    species_mapping[species]
                ]

            else:

                pred_dict[species] = 0.0

        file_results.append(pred_dict)

    return file_results

In [12]:
# =========================
# RARE SPECIES — Static priority list from rare_species_analysis.py
# Only pseudo-label chunks that contain at least one of these species
# =========================

# CRITICAL — zero training samples (threshold 0.30)
CRITICAL_SPECIES = {
    '517063', '1491113', '25073',           # Amphibia
    '47158son01', '47158son02', '47158son03', '47158son04', '47158son05',
    '47158son06', '47158son07', '47158son08', '47158son09', '47158son10',
    '47158son11', '47158son12', '47158son13', '47158son14', '47158son15',
    '47158son16', '47158son17', '47158son18', '47158son19', '47158son20',
    '47158son21', '47158son22', '47158son23', '47158son24', '47158son25',  # Insecta
}

# HIGH — fewer than 10 training samples (threshold 0.50)
HIGH_PRIORITY_SPECIES = {
    '516975', '23150', '116570', '23724',   # Mammalia + Reptilia + Amphibia (1 sample)
    '70711', '24321', '209233',             # 2 samples
    'sptnig1', '74580', '64898', '555123',
    '476521', '25214',                      # 3 samples
    '23176',                                # 4 samples
    '738183', '66971', '23154', '1595929',  # 5 samples
    '67252', '22985', '22961', '22930',     # 6 samples
    '760266',                               # 7 samples
    '22967',                                # 8 samples
    '555145',                               # 9 samples
}

# Combined set of all rare species to target
RARE_SPECIES = CRITICAL_SPECIES | HIGH_PRIORITY_SPECIES

In [13]:
PSEUDO_LABEL_THRESHOLD = 0.8
MAX_LABELS_PER_CHUNK = 3

def predict_file(file_path):

    raw_probs, chunk_npy_files = (
        get_ensemble_probs(file_path)
    )

    smoothed_rows = aggregate_and_format(
        raw_probs,
        file_path.stem,
        target_species,
        mapping_234
    )

    manifest_rows = []

    for chunk_idx, row in enumerate(smoothed_rows):

        # ----------------------------------------
        # Collect species probabilities
        # ----------------------------------------
        species_probs = [
            (
                species,
                row.get(species, 0.0)
            )
            for species in target_species
        ]

        # Highest confidence first
        species_probs.sort(
            key=lambda x: x[1],
            reverse=True
        )

        # Keep only confident predictions
        labels = [
            species
            for species, prob in species_probs
            if prob >= PSEUDO_LABEL_THRESHOLD
        ]

        # Optional cap on number of labels
        labels = labels[:MAX_LABELS_PER_CHUNK]

        if not labels:
            continue

        # Only keep chunks that contain at least one rare species
        # No point pseudo-labeling chunks with only common species
        if not any(label in RARE_SPECIES for label in labels):
            continue

        start_sec = chunk_idx * 5
        end_sec = start_sec + 5

        manifest_rows.append({
            "filename":
                chunk_npy_files[chunk_idx],
            "start":
                f"00:00:{start_sec:02d}",
            "end":
                f"00:00:{end_sec:02d}",
            "primary_label":
                ";".join(labels)
        })

    gc.collect()

    return manifest_rows

In [14]:
from pathlib import Path

train_labels_df = pd.read_csv(
    "/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv"
)



TRAIN_SOUNDSCAPE_DIR = Path(
    "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
)

existing_files = set(
    train_labels_df["filename"].unique()
)

print(
    f"Found {len(existing_files):,} labelled soundscapes"
)

candidate_files = []

for file_path in TRAIN_SOUNDSCAPE_DIR.glob("*.ogg"):


    # Only S08 S22 soundscapes
    if "_S22_" not in file_path.name and "_S08_" not in file_path.name:
        continue

    # Skip anything already labelled
    if file_path.name in existing_files:
        continue

    candidate_files.append(file_path)

print(
    f"Found {len(candidate_files):,} unlabelled S22 soundscapes"
)

all_labelled_rows = []
for file in tqdm(candidate_files):
    all_labelled_rows.extend(predict_file(file))



Found 66 labelled soundscapes
Found 3,343 unlabelled S22 soundscapes


100%|██████████| 3343/3343 [46:02<00:00,  1.21it/s]


In [15]:
# ============================================================
# SAVE MANIFEST
# ============================================================

manifest_df = pd.DataFrame(all_labelled_rows)

manifest_path = OUTPUT_DIR / "0000_soundscape_manifest.csv"

manifest_df.to_csv(
    manifest_path,
    index=False
)

print(
    f"Saved manifest with "
    f"{len(manifest_df):,} rows"
)

print(manifest_path)

# ============================================================
# UPLOAD DATASET USING KAGGLEHUB
# ============================================================

import time
import kagglehub

start = time.time()

try:
    result = kagglehub.dataset_upload(
        "gany24558/birdclef2026-s22-pseudolabels",
        str(OUTPUT_DIR),
        version_notes="Pseudo-labelled S22 soundscapes generated from B3+B0 ensemble"
    )

    print(
        f"Upload completed in "
        f"{time.time() - start:.1f}s"
    )

    print(result)
except Exception as e:
    print("UPLOAD FAILED")
    print(type(e))
    print(e)



Saved manifest with 15,951 rows
/kaggle/working/birdclef2026_s22_pseudolabels/0000_soundscape_manifest.csv
Uploading Dataset https://api.kaggle.com/datasets/gany24558/birdclef2026-s22-pseudolabels ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpwst604ex/archive.zip


Uploading: 100%|██████████| 3.23G/3.23G [00:38<00:00, 84.2MB/s]

Upload successful: /tmp/tmpwst604ex/archive.zip (3GB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/gany24558/birdclef2026-s22-pseudolabels
Upload completed in 54.5s
None


# Diagnostics

In [16]:
"""
label_counts = []

for file in candidate_files[:50]:
    rows = predict_file(file)

    for row in rows:
        label_counts.append(
            len(row["primary_label"].split(";"))
        )

print("Chunks:", len(label_counts))
print("Mean labels/chunk:", np.mean(label_counts))
print("Median:", np.median(label_counts))
print("Max:", np.max(label_counts))
"""


'\nlabel_counts = []\n\nfor file in candidate_files[:50]:\n    rows = predict_file(file)\n\n    for row in rows:\n        label_counts.append(\n            len(row["primary_label"].split(";"))\n        )\n\nprint("Chunks:", len(label_counts))\nprint("Mean labels/chunk:", np.mean(label_counts))\nprint("Median:", np.median(label_counts))\nprint("Max:", np.max(label_counts))\n'

In [17]:

"""
total_chunks = 0
kept_chunks = 0

for file in candidate_files[:500]:
    raw_probs, _ = get_ensemble_probs(file)

    total_chunks += len(raw_probs)

    rows = predict_file(file)
    kept_chunks += len(rows)

print(f"Kept {kept_chunks:,}/{total_chunks:,} chunks")
print(f"Retention = {100 * kept_chunks / total_chunks:.1f}%")

label_counts = []

for row in all_labelled_rows:
    label_counts.append(
        len(row["primary_label"].split(";"))
    )

print("Mean :", np.mean(label_counts))
print("Median :", np.median(label_counts))
print("95th :", np.percentile(label_counts, 95))
print("Max :", np.max(label_counts))

from collections import Counter

species_counter = Counter()

for row in all_labelled_rows:
    species_counter.update(
        row["primary_label"].split(";")
    )

print("Unique species:", len(species_counter))

for species, count in species_counter.most_common(20):
    print(species, count)

pseudo_species = set(species_counter.keys())

print(
    f"Pseudo species: "
    f"{len(pseudo_species)} / 234"
)

top_scores = []

for file in candidate_files[:100]:

    raw_probs, _ = get_ensemble_probs(file)

    for chunk in raw_probs:
        top_scores.append(chunk.max())

print(np.percentile(
    top_scores,
    [50, 75, 90, 95, 99]
))


print("Chunks retained:", kept_chunks)
print("Retention %:", retention)
print("Unique species:", len(pseudo_species))
print("Mean labels/chunk:", np.mean(label_counts))
print("Median labels/chunk:", np.median(label_counts))
print("Top 10 species:")
print(species_counter.most_common(10))
print("Top score percentiles:")
print(np.percentile(top_scores, [50,75,90,95,99]))        

"""



'\ntotal_chunks = 0\nkept_chunks = 0\n\nfor file in candidate_files[:500]:\n    raw_probs, _ = get_ensemble_probs(file)\n\n    total_chunks += len(raw_probs)\n\n    rows = predict_file(file)\n    kept_chunks += len(rows)\n\nprint(f"Kept {kept_chunks:,}/{total_chunks:,} chunks")\nprint(f"Retention = {100 * kept_chunks / total_chunks:.1f}%")\n\nlabel_counts = []\n\nfor row in all_labelled_rows:\n    label_counts.append(\n        len(row["primary_label"].split(";"))\n    )\n\nprint("Mean :", np.mean(label_counts))\nprint("Median :", np.median(label_counts))\nprint("95th :", np.percentile(label_counts, 95))\nprint("Max :", np.max(label_counts))\n\nfrom collections import Counter\n\nspecies_counter = Counter()\n\nfor row in all_labelled_rows:\n    species_counter.update(\n        row["primary_label"].split(";")\n    )\n\nprint("Unique species:", len(species_counter))\n\nfor species, count in species_counter.most_common(20):\n    print(species, count)\n\npseudo_species = set(species_counter.